In [42]:
import os, ast, config, logging
import numpy as np
import pandas as pd


# LOGGING SETUP

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("Data-Analyst")

In [39]:
def load_metadata(data_dir: str) -> pd.DataFrame:

    csv_path = os.path.join(data_dir, "ptbxl_database.csv")
    log.info(f"Loading metadata from {csv_path}...")
    
    meta = pd.read_csv(csv_path, index_col="ecg_id")
    
    meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval) # Convert stringified dict to actual dict
    
    before = len(meta)
    meta = meta.drop(index=[i for i in config.DROPPED_ECG_IDS if i in meta.index])
    log.info(f"Dropped {before - len(meta)} duplicate records. {len(meta)} records remain.")

    #Fix privacy-masked age
    masked = (meta["age"] == 300).sum()
    meta["age"] = meta["age"].replace(300, np.nan)
    log.info(f"Masked ages (300 -> NaN): {masked} patients aged >89.")

    return meta


    

In [40]:
def load_scp_statements(data_dir: str) -> pd.DataFrame:
    scp_path = os.path.join(data_dir, "scp_statements.csv")
    scp = pd.read_csv(scp_path, index_col=0)
    
    # Filter only diagnostic statements
    scp_diag = scp[scp["diagnostic"] == 1].copy()
    
    log.info(f"SCP diagnostic codes loaded: {len(scp_diag)} codes "
             f"across {scp_diag['diagnostic_class'].nunique()} superclasses.")
    
    return scp_diag


In [41]:
def inspect_missing_metadata(meta: pd.DataFrame) -> None:
    missing = meta.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    print("\n[2] Metadata missing values:")
    if missing.empty:
        print("None.")
    else:
        for col, cnt in missing.items():
            pct = 100 * cnt / len(meta)
            print(f"{col:<25} {cnt:>5} ({pct:.1f}%)")